# Closing Two Open Evidence Gaps

Two claims were made about why merged adapters underperform on GSM8K, but never actually confirmed with real numbers:

1. **The SLERP norm-inflation bug was fixed, but never re-evaluated.** The fix was verified on raw delta tensors (norm ratio corrected from 1.13 to 0.90, cosine similarity unchanged at 0.63) — but the last GSM8K eval run after the fix showed identical numbers to the pre-fix run, because it was still hitting stale checkpoints. This notebook re-evaluates the actually-fixed model.

2. **SVD/BWSum's rank-16 truncation and TIES/DARE's element-trim were claimed to destroy information, but never quantified.** This notebook measures exactly how much singular-value energy each truncation strategy actually keeps, on the real adapter deltas.

In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"


## Part 1 — Does the SLERP Fix Actually Improve GSM8K?

Evaluates the fixed checkpoint (geometric-mean norm interpolation) fresh, and compares directly against the pre-fix numbers.

In [2]:
import torch
import gc
import re
import math
import multiprocessing
import contextlib
import io
from datasets import load_dataset
from tqdm import tqdm
from unsloth import FastLanguageModel

def build_chat_prompt(tokenizer, user_content: str) -> str:
    messages = [{"role": "user", "content": user_content}]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False,
    )

def prep_tokenizer_for_generation(tokenizer):
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    return tokenizer

# ── GSM8K ──
def extract_gsm8k_answer(text: str) -> str:
    hash_match = re.findall(r"####\s*(-?[\d,]+\.?\d*)", text)
    if hash_match:
        return hash_match[-1].replace(",", "").strip()
    boxed = re.findall(r"\\boxed\{([^}]+)\}", text)
    if boxed:
        return boxed[-1].replace(",", "").strip()
    numbers = re.findall(r"-?\d[\d,]*\.?\d*", text)
    if numbers:
        return numbers[-1].replace(",", "").strip()
    return text.strip()

def format_gsm8k_prompt(question: str) -> str:
    return (f"Solve the following math problem. Show your reasoning and put "
            f"your final numeric answer after '#### '.\n\nQuestion: {question}")

def evaluate_gsm8k(model, tokenizer, model_name="model", num_samples=200, batch_size=4,
                    max_new_tokens=320, device="cuda"):
    print(f"\n{'─'*60}\n[GSM8K] Evaluating: {model_name}\n{'─'*60}")
    model.eval()
    prep_tokenizer_for_generation(tokenizer)
    dataset = load_dataset("gsm8k", "main", split="test")
    dataset = dataset.select(range(min(num_samples, len(dataset))))
    preds, labels = [], []
    for i in tqdm(range(0, len(dataset), batch_size), desc=f"{model_name} [gsm8k]"):
        batch = dataset[i : i + batch_size]
        questions    = batch["question"]
        true_answers = [extract_gsm8k_answer(a) for a in batch["answer"]]
        prompts      = [build_chat_prompt(tokenizer, format_gsm8k_prompt(q)) for q in questions]
        inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True,
                            max_length=512, add_special_tokens=False).to(device)
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                                      pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id,
                                      repetition_penalty=1.3)
        input_len = inputs["input_ids"].shape[1]
        for j, output in enumerate(outputs):
            generated = tokenizer.decode(output[input_len:], skip_special_tokens=True)
            preds.append(extract_gsm8k_answer(generated))
            labels.append(true_answers[j])
    per_sample_exact = [int(p.strip() == l.strip()) for p, l in zip(preds, labels)]
    exact_match = round(sum(per_sample_exact) / len(per_sample_exact), 4)
    print(f"  Exact Match: {exact_match:.4f}")
    return {"repo_id": model_name, "exact_match": exact_match, "num_samples": len(per_sample_exact),
            "per_sample_exact": per_sample_exact}

# ── HumanEval ──
def extract_code(generated: str, problem_prompt: str, entry_point: str) -> str:
    text = generated.strip()
    fence = re.search(r"```(?:python)?\s*\n?(.*?)```", text, re.DOTALL)
    if fence:
        text = fence.group(1).strip()
    if f"def {entry_point}" in text:
        return text
    return problem_prompt + "\n" + text

def _unsafe_execute(program: str, result_list, timeout: int):
    import signal
    def handler(signum, frame):
        raise TimeoutError("execution timed out")
    try:
        signal.signal(signal.SIGALRM, handler)
        signal.alarm(timeout)
        exec_globals = {}
        with contextlib.redirect_stdout(io.StringIO()):
            exec(program, exec_globals)
        signal.alarm(0)
        result_list.append("passed")
    except Exception as e:
        result_list.append(f"failed: {type(e).__name__}: {e}")

def check_correctness(problem: dict, completion_code: str, timeout: int = 5) -> bool:
    program = completion_code + "\n" + problem["test"] + f"\ncheck({problem['entry_point']})\n"
    manager = multiprocessing.Manager()
    result_list = manager.list()
    p = multiprocessing.Process(target=_unsafe_execute, args=(program, result_list, timeout))
    p.start()
    p.join(timeout=timeout + 1)
    if p.is_alive():
        p.kill(); p.join()
    if not result_list:
        result_list.append("failed: timeout")
    return result_list[0] == "passed"

def format_humaneval_prompt(problem_prompt: str) -> str:
    return ("Complete the following Python function. Return ONLY the complete "
            "function code (including the signature), with no explanations and "
            f"no markdown formatting.\n\n{problem_prompt}")

def evaluate_humaneval(model, tokenizer, model_name="model", num_samples=164, batch_size=4,
                        max_new_tokens=384, device="cuda"):
    print(f"\n{'─'*60}\n[HumanEval] Evaluating: {model_name}\n{'─'*60}")
    model.eval()
    prep_tokenizer_for_generation(tokenizer)
    dataset = load_dataset("openai/openai_humaneval", split="test")
    dataset = dataset.select(range(min(num_samples, len(dataset))))
    per_sample_pass = []
    for i in tqdm(range(0, len(dataset), batch_size), desc=f"{model_name} [humaneval]"):
        batch = dataset[i : i + batch_size]
        problem_prompts = batch["prompt"]
        entry_points    = batch["entry_point"]
        prompts = [build_chat_prompt(tokenizer, format_humaneval_prompt(p)) for p in problem_prompts]
        inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True,
                            max_length=768, add_special_tokens=False).to(device)
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                                      pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id,
                                      repetition_penalty=1.1)
        input_len = inputs["input_ids"].shape[1]
        for j, output in enumerate(outputs):
            generated = tokenizer.decode(output[input_len:], skip_special_tokens=True)
            code = extract_code(generated, problem_prompts[j], entry_points[j])
            problem = {"prompt": problem_prompts[j], "test": batch["test"][j], "entry_point": entry_points[j]}
            per_sample_pass.append(int(check_correctness(problem, code, timeout=5)))
    pass_at_1 = round(sum(per_sample_pass) / len(per_sample_pass), 4)
    print(f"  pass@1: {pass_at_1:.4f}")
    return {"repo_id": model_name, "pass_at_1": pass_at_1, "num_samples": len(per_sample_pass),
            "per_sample_pass": per_sample_pass}

# ── Dolly-15k perplexity ──
def format_dolly_prompt(instruction: str, context: str) -> str:
    if context:
        return f"Instruction: {instruction}\nContext: {context}\nResponse:"
    return f"Instruction: {instruction}\nResponse:"

def evaluate_dolly_perplexity(model, tokenizer, model_name="model", num_samples=200,
                               max_length=512, device="cuda"):
    print(f"\n{'─'*60}\n[Dolly-PPL] Evaluating: {model_name}\n{'─'*60}")
    model.eval()
    dataset = load_dataset("databricks/databricks-dolly-15k", split="train")
    dataset = dataset.shuffle(seed=42).select(range(min(num_samples, len(dataset))))
    per_sample_nll, per_sample_ppl = [], []
    for ex in tqdm(dataset, desc=f"{model_name} [dolly-ppl]"):
        prompt = format_dolly_prompt(ex["instruction"], ex.get("context", ""))
        response = ex["response"]
        if not response.strip():
            continue
        full_text = prompt + " " + response
        prompt_ids = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=max_length).input_ids.to(device)
        full_ids   = tokenizer(full_text, return_tensors="pt", truncation=True, max_length=max_length).input_ids.to(device)
        if full_ids.shape[1] <= prompt_ids.shape[1]:
            continue
        labels = full_ids.clone()
        labels[:, : prompt_ids.shape[1]] = -100
        with torch.no_grad():
            out = model(full_ids, labels=labels)
        nll = out.loss.item()
        per_sample_nll.append(nll)
        per_sample_ppl.append(math.exp(nll))
    result = {"repo_id": model_name, "perplexity": round(sum(per_sample_ppl) / len(per_sample_ppl), 4),
              "mean_nll": round(sum(per_sample_nll) / len(per_sample_nll), 4),
              "num_samples": len(per_sample_ppl), "per_sample_nll": per_sample_nll}
    print(f"  Perplexity: {result['perplexity']:.4f}  (mean NLL: {result['mean_nll']:.4f})")
    return result


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [3]:
from unsloth import FastLanguageModel
import torch, gc

# This repo was overwritten IN PLACE with the geometric-mean-fixed slerp_merge —
# same repo name as the old buggy version, so this pulls the fixed weights.
FIXED_SLERP_REPO = "Srishtik/Qwen3-0.6B-slerp-order-dolly-metamath-codealpaca-3-adapters-merged-2"

eval_model, eval_tokenizer = FastLanguageModel.from_pretrained(
    model_name     = FIXED_SLERP_REPO,
    max_seq_length = 1024,
    load_in_4bit   = True,
    dtype          = torch.float16,
)
FastLanguageModel.for_inference(eval_model)

fixed_slerp_gsm8k     = evaluate_gsm8k(eval_model, eval_tokenizer, model_name=FIXED_SLERP_REPO,
                                        num_samples=200, batch_size=4)
fixed_slerp_humaneval = evaluate_humaneval(eval_model, eval_tokenizer, model_name=FIXED_SLERP_REPO,
                                            num_samples=164, batch_size=4)
fixed_slerp_dolly     = evaluate_dolly_perplexity(eval_model, eval_tokenizer, model_name=FIXED_SLERP_REPO,
                                                   num_samples=200)

del eval_model, eval_tokenizer
gc.collect()
torch.cuda.empty_cache()

# ── Before/after comparison ──
old_buggy = {"gsm8k": 0.0100, "humaneval": 0.1463, "dolly_ppl": 13.0035}  # pre-fix, same repo/order

print(f"\n{'═'*70}")
print(f"SLERP (dolly→metamath→codealpaca order) — BEFORE vs AFTER the norm fix")
print(f"{'═'*70}")
print(f"{'Metric':<18}{'Before (buggy)':>18}{'After (fixed)':>18}{'Change':>14}")
print("-" * 70)
print(f"{'GSM8K EM':<18}{old_buggy['gsm8k']:>18.4f}{fixed_slerp_gsm8k['exact_match']:>18.4f}"
      f"{fixed_slerp_gsm8k['exact_match'] - old_buggy['gsm8k']:>+14.4f}")
print(f"{'HumanEval p@1':<18}{old_buggy['humaneval']:>18.4f}{fixed_slerp_humaneval['pass_at_1']:>18.4f}"
      f"{fixed_slerp_humaneval['pass_at_1'] - old_buggy['humaneval']:>+14.4f}")
print(f"{'Dolly PPL':<18}{old_buggy['dolly_ppl']:>18.4f}{fixed_slerp_dolly['perplexity']:>18.4f}"
      f"{fixed_slerp_dolly['perplexity'] - old_buggy['dolly_ppl']:>+14.4f}")
print(f"{'═'*70}")

print("\nFor reference — where this now stands against the other 6 methods on GSM8K:")
other_methods_gsm8k = {"linear": 0.1100, "svd": 0.1550, "ties": 0.0550, "dare": 0.0400, "bwsum": 0.1200}
for name, score in sorted(other_methods_gsm8k.items(), key=lambda x: -x[1]):
    print(f"  {name:<12} {score:.4f}")
print(f"  {'slerp (fixed)':<12} {fixed_slerp_gsm8k['exact_match']:.4f}  <-- this run")


==((====))==  Unsloth 2026.7.5: Fast Qwen3 patching. Transformers: 5.0.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]


────────────────────────────────────────────────────────────
[GSM8K] Evaluating: Srishtik/Qwen3-0.6B-slerp-order-dolly-metamath-codealpaca-3-adapters-merged-2
────────────────────────────────────────────────────────────


README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

...

README.md: 0.00B [00:00, ?B/s]

databricks-dolly-15k.jsonl:   0%|          | 0.00/13.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/15011 [00:00<?, ? examples/s]

Srishtik/Qwen3-0.6B-slerp-order-dolly-metamath-codealpaca-3-adapters-merged-2 [dolly-ppl]: 100%|██████████| 200/200 [00:25<00:00,  7.88it/s]


  Perplexity: 13.0035  (mean NLL: 1.9500)

══════════════════════════════════════════════════════════════════════
SLERP (dolly→metamath→codealpaca order) — BEFORE vs AFTER the norm fix
══════════════════════════════════════════════════════════════════════
Metric                Before (buggy)     After (fixed)        Change
----------------------------------------------------------------------
GSM8K EM                      0.0100            0.0100       +0.0000
HumanEval p@1                 0.1463            0.1463       +0.0000
Dolly PPL                    13.0035           13.0035       +0.0000
══════════════════════════════════════════════════════════════════════

For reference — where this now stands against the other 6 methods on GSM8K:
  svd          0.1550
  bwsum        0.1200
  linear       0.1100
  ties         0.0550
  dare         0.0400
  slerp (fixed) 0.0100  <-- this run


## Part 2 — Quantifying SVD/BWSum's Rank Truncation and TIES/DARE's Element Trim

Measures singular-value energy retained under each method's actual truncation budget, on the real combined 3-adapter delta, layer by layer.

In [4]:
from safetensors.torch import load_file
from huggingface_hub import hf_hub_download
import json
import torch

def get_lora_deltas(repo_name: str, hf_username: str = "Srishtik", lora_alpha: int = None, r: int = None) -> dict:
    repo_id = f"{hf_username}/{repo_name}"
    config_path = hf_hub_download(repo_id=repo_id, filename="adapter_config.json")
    cfg = json.load(open(config_path))
    actual_r     = r if r is not None else cfg.get("r")
    actual_alpha = lora_alpha if lora_alpha is not None else cfg.get("lora_alpha")
    path = hf_hub_download(repo_id=repo_id, filename="adapter_model.safetensors")
    adapter_weights = load_file(path)
    scale = actual_alpha / actual_r
    layers = {}
    for key, val in adapter_weights.items():
        if "lora_A" in key:
            base_key = key.replace("lora_A.default.weight", "").replace("lora_A.weight", "")
            layers.setdefault(base_key, {})["A"] = val.float()
        elif "lora_B" in key:
            base_key = key.replace("lora_B.default.weight", "").replace("lora_B.weight", "")
            layers.setdefault(base_key, {})["B"] = val.float()
    deltas = {}
    for base_key, mats in layers.items():
        if "A" in mats and "B" in mats:
            deltas[base_key] = scale * (mats["B"] @ mats["A"])
    print(f"  {repo_name:<35} r={actual_r}  alpha={actual_alpha}  scale={scale:.4f}")
    return deltas

print("Loading adapter deltas for rank/trim analysis:")
dolly_deltas      = get_lora_deltas("qwen3-trained-on-dolly-15k")
metamath_deltas   = get_lora_deltas("qwen3-trained-on-metamath-15k")
codealpaca_deltas = get_lora_deltas("qwen3-trained-on-code-alpaca-18k")


Loading adapter deltas for rank/trim analysis:


adapter_config.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/40.4M [00:00<?, ?B/s]

  qwen3-trained-on-dolly-15k          r=16  alpha=32  scale=2.0000


adapter_config.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/40.4M [00:00<?, ?B/s]

  qwen3-trained-on-metamath-15k       r=16  alpha=32  scale=2.0000


adapter_config.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/40.4M [00:00<?, ?B/s]

  qwen3-trained-on-code-alpaca-18k    r=16  alpha=32  scale=2.0000


In [5]:
import torch

def svd_energy_retained(matrix: torch.Tensor, rank: int) -> float:
    U, S, V = torch.linalg.svd(matrix.float(), full_matrices=False)
    total = (S ** 2).sum()
    kept  = (S[:rank] ** 2).sum()
    return (kept / total).item()

def elementwise_trim_energy_retained(matrix: torch.Tensor, density: float) -> float:
    flat = matrix.abs().flatten()
    k = max(1, int(density * flat.numel()))
    threshold = torch.topk(flat, k).values.min()
    mask = matrix.abs() >= threshold
    trimmed = matrix * mask
    orig_energy = (matrix.float() ** 2).sum()
    kept_energy = (trimmed.float() ** 2).sum()
    return (kept_energy / orig_energy).item()


keys = list(set(dolly_deltas.keys()) & set(metamath_deltas.keys()) & set(codealpaca_deltas.keys()))

svd_r16_energies  = []  # what SVD/BWSum actually keep: combined matrix truncated to shared rank-16
svd_r48_energies  = []  # what NO truncation would keep: combined matrix at full rank (sum of individual ranks)
trim_d02_energies = []  # what TIES/DARE keep: top-20% elements by magnitude, on the same combined matrix

for key in keys:
    combined = dolly_deltas[key] + metamath_deltas[key] + codealpaca_deltas[key]
    svd_r16_energies.append(svd_energy_retained(combined, 16))
    svd_r48_energies.append(svd_energy_retained(combined, 48))
    trim_d02_energies.append(elementwise_trim_energy_retained(combined, 0.2))

def summarize(name, vals):
    vals_t = torch.tensor(vals)
    print(f"{name:<45} mean={vals_t.mean():.4f}  std={vals_t.std():.4f}  "
          f"min={vals_t.min():.4f}  max={vals_t.max():.4f}")

print(f"Energy retained on the COMBINED (summed) 3-adapter delta, across {len(keys)} layers:\n")
summarize("SVD/BWSum's actual budget (rank=16, shared)", svd_r16_energies)
summarize("No truncation (rank=48, sum of individual ranks)", svd_r48_energies)
summarize("TIES/DARE's actual budget (top 20% elements)", trim_d02_energies)

print(f"\nEnergy LOST specifically to sharing rank-16 across 3 tasks "
      f"(rank-48 energy minus rank-16 energy, i.e. what a wider budget would have kept):")
lost_to_sharing = torch.tensor(svd_r48_energies) - torch.tensor(svd_r16_energies)
print(f"  mean={lost_to_sharing.mean():.4f}  (as a fraction of total signal energy, averaged across layers)")

print(f"\nWhich is more destructive on this data — SVD's rank-16 truncation or TIES/DARE's element trim?")
svd_mean  = torch.tensor(svd_r16_energies).mean().item()
trim_mean = torch.tensor(trim_d02_energies).mean().item()
if svd_mean > trim_mean:
    print(f"  SVD (rank-16) retains MORE energy ({svd_mean:.4f}) than element-trim ({trim_mean:.4f}) —")
    print(f"  TIES/DARE's magnitude-based trim is the more destructive truncation of the two.")
else:
    print(f"  Element-trim retains MORE energy ({trim_mean:.4f}) than SVD rank-16 ({svd_mean:.4f}) —")
    print(f"  the shared rank-16 budget is the more destructive constraint of the two.")

# ── Worst-affected layers under SVD's rank-16 budget ──
print("\nLayers where SVD's rank-16 truncation loses the most energy:")
layer_loss = {key: svd_r48_energies[i] - svd_r16_energies[i] for i, key in enumerate(keys)}
worst = sorted(layer_loss.items(), key=lambda x: -x[1])[:10]
for key, loss in worst:
    print(f"  {key:<50} energy lost={loss:.4f}")


Energy retained on the COMBINED (summed) 3-adapter delta, across 196 layers:

SVD/BWSum's actual budget (rank=16, shared)   mean=0.9493  std=0.0141  min=0.9079  max=0.9801
No truncation (rank=48, sum of individual ranks) mean=1.0000  std=0.0000  min=1.0000  max=1.0000
TIES/DARE's actual budget (top 20% elements)  mean=0.6906  std=0.0201  min=0.6685  max=0.8207

Energy LOST specifically to sharing rank-16 across 3 tasks (rank-48 energy minus rank-16 energy, i.e. what a wider budget would have kept):
  mean=0.0507  (as a fraction of total signal energy, averaged across layers)

Which is more destructive on this data — SVD's rank-16 truncation or TIES/DARE's element trim?
  SVD (rank-16) retains MORE energy (0.9493) than element-trim (0.6906) —
  TIES/DARE's magnitude-based trim is the more destructive truncation of the two.

Layers where SVD's rank-16 truncation loses the most energy:
  base_model.model.model.layers.9.mlp.down_proj.     energy lost=0.0921
  base_model.model.model.layers.